# Cnn And Matching
Original workflow: `CNN&MN_baseline.ipynb`. Imports now use the `amgpn` package.
Run from the repository root after an editable install. Configuration keys retain the original notebook/cell identifiers.


In [ ]:
from amgpn.config import require as _cfg_require, resolve as _cfg_resolve, format_value as _cfg_format
#CNN
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import numpy as np
from tqdm import tqdm
from amgpn.data.wavelet_episodes import *
from amgpn.comparisons.cnn import *
import higher
from amgpn.visualization.single import *
from thop import profile, clever_format
# ==================== 使用示例 ====================
N_WAY = _cfg_require('CNN&MN_baseline.ipynb.cell0.module.N_WAY')
K_SHOT = _cfg_require('CNN&MN_baseline.ipynb.cell0.module.K_SHOT')
Q_QUERY = _cfg_require('CNN&MN_baseline.ipynb.cell0.module.Q_QUERY')
NUM_TASKS = _cfg_require('CNN&MN_baseline.ipynb.cell0.module.NUM_TASKS')
num_epochs = _cfg_require('CNN&MN_baseline.ipynb.cell0.module.num_epochs')
SEED=_cfg_require('CNN&MN_baseline.ipynb.cell0.module.SEED')
# Historical configuration omitted; provide values through private configuration.
dataset_path = _cfg_require('CNN&MN_baseline.ipynb.cell0.module.dataset_path')

if __name__ == "__main__":

    
    train_dataset=UAVDataset(data_dir_path=dataset_path,seed=SEED,is_train=True,train_ratio=_cfg_require('CNN&MN_baseline.ipynb.cell0.module.train_ratio'),max_sample_count=_cfg_require('CNN&MN_baseline.ipynb.cell0.module.max_sample_count'),
                            )
    test_dataset=UAVDataset(data_dir_path=dataset_path,seed=SEED,is_train=False,train_ratio=_cfg_require('CNN&MN_baseline.ipynb.cell0.module.train_ratio__2'),max_sample_count=_cfg_require('CNN&MN_baseline.ipynb.cell0.module.max_sample_count__2'),
                            )

    meta_train_dataset = MetaDataset(train_dataset, 8,15,5,num_tasks=NUM_TASKS)
    meta_test_dataset = MetaDataset(test_dataset, N_WAY, K_SHOT, Q_QUERY,num_tasks=_cfg_require('CNN&MN_baseline.ipynb.cell0.module.num_tasks'))
    # DataLoader
    train_loader = DataLoader(meta_train_dataset, batch_size=_cfg_require('CNN&MN_baseline.ipynb.cell0.module.batch_size'), shuffle=True, drop_last=True, num_workers=_cfg_require('CNN&MN_baseline.ipynb.cell0.module.num_workers'))
    test_loader = DataLoader(meta_test_dataset, batch_size=_cfg_require('CNN&MN_baseline.ipynb.cell0.module.batch_size__2'), shuffle=True)

    # 初始化设备
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # 创建CNN baseline模型
    model = CNN_Baseline_Simple(feature_dim=_cfg_require('CNN&MN_baseline.ipynb.cell0.module.feature_dim'))
    count_parameters(model)
    
#     params = count_parameters(model)
    
# Historical configuration omitted; provide values through private configuration.
#     # 前向传播
#     v, s = model(x)
#     print(f"Embedding features (v) shape: {v.shape}")
#     print(f"Precision features (s) shape: {s.shape}")
# Historical configuration omitted; provide values through private configuration.

# Historical configuration omitted; provide values through private configuration.
    
#     flops, params = profile(model, inputs=(input_tensor,))
# Historical configuration omitted; provide values through private configuration.
#     print(f"FLOPs: {flops}, Params: {params}")
    # 创建训练器
    meta_learner = CNNBaselineTrainer(
        model=model,
        device=device,
        mlr=_cfg_require('CNN&MN_baseline.ipynb.cell0.module.mlr'),
        crop_ratio_h=_cfg_require('CNN&MN_baseline.ipynb.cell0.module.crop_ratio_h'),  # 根据需要设置
        crop_ratio_l=_cfg_require('CNN&MN_baseline.ipynb.cell0.module.crop_ratio_l'),
        resize=_cfg_require('CNN&MN_baseline.ipynb.cell0.module.resize'),
        save_middle=_cfg_require('CNN&MN_baseline.ipynb.cell0.module.save_middle')
    )

    print("开始训练...")
    meta_learner.train(train_loader, test_loader, num_epochs, save_path=_cfg_require('CNN&MN_baseline.ipynb.cell0.module.save_path'))
    print("训练完成！")

    # 清理内存
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    # 评估
    meta_learner.load_model(_cfg_require('CNN&MN_baseline.ipynb.cell0.module.path'))
    test_dataset=UAVDataset(data_dir_path=dataset_path,seed=SEED,is_train=False,
                        train_ratio=_cfg_require('CNN&MN_baseline.ipynb.cell0.module.train_ratio__3'),max_sample_count=_cfg_require('CNN&MN_baseline.ipynb.cell0.module.max_sample_count__3'),
                        )
    results = meta_learner.full_evaluation(test_dataset, n_trials=_cfg_require('CNN&MN_baseline.ipynb.cell0.module.n_trials'), q_query=_cfg_require('CNN&MN_baseline.ipynb.cell0.module.q_query'))
    plot_all_visualizations(results,output_dir=_cfg_require('CNN&MN_baseline.ipynb.cell0.module.output_dir'))#V2_rnx+rnx_d_ft5:1_L_8_1_5


In [ ]:
from amgpn.config import require as _cfg_require, resolve as _cfg_resolve, format_value as _cfg_format
#MatchingNetworks
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import numpy as np
from tqdm import tqdm
from amgpn.data.wavelet_episodes import *
from amgpn.comparisons.matching import *
import higher
from amgpn.visualization.single import *
from thop import profile, clever_format
# ==================== 使用示例 ====================
N_WAY = _cfg_require('CNN&MN_baseline.ipynb.cell1.module.N_WAY')
K_SHOT = _cfg_require('CNN&MN_baseline.ipynb.cell1.module.K_SHOT')
Q_QUERY = _cfg_require('CNN&MN_baseline.ipynb.cell1.module.Q_QUERY')
NUM_TASKS = _cfg_require('CNN&MN_baseline.ipynb.cell1.module.NUM_TASKS')
num_epochs = _cfg_require('CNN&MN_baseline.ipynb.cell1.module.num_epochs')
SEED=_cfg_require('CNN&MN_baseline.ipynb.cell1.module.SEED')
# Historical configuration omitted; provide values through private configuration.
dataset_path = _cfg_require('CNN&MN_baseline.ipynb.cell1.module.dataset_path')

if __name__ == "__main__":

    
    train_dataset=UAVDataset(data_dir_path=dataset_path,seed=SEED,is_train=True,train_ratio=_cfg_require('CNN&MN_baseline.ipynb.cell1.module.train_ratio'),max_sample_count=_cfg_require('CNN&MN_baseline.ipynb.cell1.module.max_sample_count'),
                            )
    test_dataset=UAVDataset(data_dir_path=dataset_path,seed=SEED,is_train=False,train_ratio=_cfg_require('CNN&MN_baseline.ipynb.cell1.module.train_ratio__2'),max_sample_count=_cfg_require('CNN&MN_baseline.ipynb.cell1.module.max_sample_count__2'),
                            )

    meta_train_dataset = MetaDataset(train_dataset, 8,15,5,num_tasks=NUM_TASKS)
    meta_test_dataset = MetaDataset(test_dataset, N_WAY, K_SHOT, Q_QUERY,num_tasks=_cfg_require('CNN&MN_baseline.ipynb.cell1.module.num_tasks'))
    # DataLoader
    train_loader = DataLoader(meta_train_dataset, batch_size=_cfg_require('CNN&MN_baseline.ipynb.cell1.module.batch_size'), shuffle=True, drop_last=True, num_workers=_cfg_require('CNN&MN_baseline.ipynb.cell1.module.num_workers'))
    test_loader = DataLoader(meta_test_dataset, batch_size=_cfg_require('CNN&MN_baseline.ipynb.cell1.module.batch_size__2'), shuffle=True)

    # 初始化设备
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # 创建模型
    model = MN_Backbone(feature_dim=_cfg_require('CNN&MN_baseline.ipynb.cell1.module.feature_dim'))
    count_parameters(model)
    
    # 创建训练器
    meta_learner = MatchingNetworkTrainer(
        model=model,
        device=device,
        lr=_cfg_require('CNN&MN_baseline.ipynb.cell1.module.lr'),
        crop_ratio_h=_cfg_require('CNN&MN_baseline.ipynb.cell1.module.crop_ratio_h'),
        crop_ratio_l=_cfg_require('CNN&MN_baseline.ipynb.cell1.module.crop_ratio_l'),
        resize=_cfg_require('CNN&MN_baseline.ipynb.cell1.module.resize'),
        save_middle=_cfg_require('CNN&MN_baseline.ipynb.cell1.module.save_middle')
        )

    print("开始训练...")
    # Historical configuration omitted; provide values through private configuration.
    print("训练完成！")

    # 清理内存
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    # 评估
    meta_learner.load_model(_cfg_require('CNN&MN_baseline.ipynb.cell1.module.path'))
    test_dataset=UAVDataset(data_dir_path=dataset_path,seed=SEED,is_train=False,
                        train_ratio=_cfg_require('CNN&MN_baseline.ipynb.cell1.module.train_ratio__3'),max_sample_count=_cfg_require('CNN&MN_baseline.ipynb.cell1.module.max_sample_count__3'),
                        )
    results = meta_learner.full_evaluation(test_dataset, n_trials=_cfg_require('CNN&MN_baseline.ipynb.cell1.module.n_trials'), q_query=_cfg_require('CNN&MN_baseline.ipynb.cell1.module.q_query'))
    plot_all_visualizations(results,output_dir=_cfg_require('CNN&MN_baseline.ipynb.cell1.module.output_dir'))#V2_rnx+rnx_d_ft5:1_L_8_1_5